# C0 — Data Connection & Setup
### Hạng mục C — Dự báo nhu cầu & Chiến lược | DATA EXPLORERS 2026

---
**Mục tiêu notebook này:**
- Kết nối PostgreSQL database `tnbike_db`
- Kéo dữ liệu từ bảng `fact_sales` (2025 – T3/2026)
- Tiền xử lý và lưu các dataset dùng chung cho C1, C2, C3
- Kiểm tra chất lượng dữ liệu trước khi đưa vào mô hình

---
> **Chạy notebook này TRƯỚC khi chạy C1, C2, C3**

# Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

# Database connection
import psycopg2
from sqlalchemy import create_engine

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, HTML

print('✅ Libraries loaded successfully')

# Database Configuration

In [ ]:
# ─── Cấu hình kết nối PostgreSQL ───
# Thay đổi thông tin kết nối phù hợp với môi trường của bạn
DB_CONFIG = {
    'host':     'localhost',
    'port':     5432,
    'database': 'tnbike_db',
    'user':     'postgres',
    'password': 'your_password_here'   # ← sửa lại
}

# SQLAlchemy connection string
DB_URL = f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

print(f"Connection target: {DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}")

# Connect & Test

In [ ]:
# Kiểm tra kết nối
try:
    engine = create_engine(DB_URL)
    with engine.connect() as conn:
        result = pd.read_sql("SELECT COUNT(*) AS total_rows FROM tnbike.fact_sales", conn)
        print(f"✅ Connected successfully!")
        print(f"   fact_sales rows: {result['total_rows'][0]:,}")
except Exception as e:
    print(f"❌ Connection failed: {e}")

# Pull Data from Database

In [ ]:
# ─── Kéo toàn bộ fact_sales (2025 – T3/2026) ───
query_fact = """
    SELECT
        order_date,
        fiscal_year,
        fiscal_quarter,
        fiscal_month,
        week_of_year,
        so_number,
        customer_code,
        customer_name,
        province_name,
        region,
        product_code,
        product_name,
        color,
        line_name,
        group_code,
        group_name,
        quantity,
        unit_price,
        line_total
    FROM tnbike.fact_sales
    ORDER BY order_date
"""

df_fact = pd.read_sql(query_fact, engine)
df_fact['order_date'] = pd.to_datetime(df_fact['order_date'])

print(f"Rows loaded: {len(df_fact):,}")
df_fact.head(3)

In [ ]:
# ─── Kéo bảng customer ───
query_cust = """
    SELECT
        c.customer_code,
        c.customer_name,
        c.customer_tier,
        c.is_active,
        p.province_name,
        p.region
    FROM tnbike.customer c
    LEFT JOIN tnbike.province p ON p.province_id = c.province_id
"""

df_customers = pd.read_sql(query_cust, engine)
print(f"Customers loaded: {len(df_customers):,}")
df_customers.head(3)

# Data Preprocessing

In [ ]:
# ─── Kiểm tra phạm vi thời gian ───
print(f"Date range : {df_fact['order_date'].min().date()} → {df_fact['order_date'].max().date()}")
print(f"Months     : {df_fact['order_date'].dt.to_period('M').nunique()}")
print(f"Products   : {df_fact['product_code'].nunique()}")
print(f"Customers  : {df_fact['customer_code'].nunique()}")
print(f"Groups     : {df_fact['group_code'].unique().tolist()}")

In [ ]:
# ─── Tổng hợp doanh số theo tháng (dùng cho C1) ───
df_monthly = (
    df_fact
    .groupby(['fiscal_year', 'fiscal_month', 'group_code', 'group_name'])
    .agg(
        total_revenue = ('line_total', 'sum'),
        total_qty     = ('quantity',   'sum'),
        order_count   = ('so_number',  'nunique')
    )
    .reset_index()
)

# Tạo cột date chuẩn YYYY-MM-01
df_monthly['ds'] = pd.to_datetime(
    df_monthly['fiscal_year'].astype(str) + '-' +
    df_monthly['fiscal_month'].astype(str).str.zfill(2) + '-01'
)

print(f"Monthly rows: {len(df_monthly):,}")
df_monthly.sort_values('ds').tail(6)

In [ ]:
# ─── Tổng hợp theo SKU × màu × tháng (dùng cho C2) ───
df_sku_monthly = (
    df_fact
    .groupby(['fiscal_year', 'fiscal_month', 'product_code', 'product_name',
              'color', 'line_name', 'group_code'])
    .agg(
        total_qty     = ('quantity',  'sum'),
        total_revenue = ('line_total','sum')
    )
    .reset_index()
)
df_sku_monthly['ds'] = pd.to_datetime(
    df_sku_monthly['fiscal_year'].astype(str) + '-' +
    df_sku_monthly['fiscal_month'].astype(str).str.zfill(2) + '-01'
)
print(f"SKU-monthly rows: {len(df_sku_monthly):,}")
df_sku_monthly.tail(3)

In [ ]:
# ─── Tổng hợp theo khách hàng × tháng (dùng cho C3) ───
df_cust_monthly = (
    df_fact
    .groupby(['fiscal_year', 'fiscal_month', 'customer_code', 'customer_name',
              'province_name', 'region'])
    .agg(
        total_revenue = ('line_total', 'sum'),
        total_qty     = ('quantity',   'sum'),
        order_count   = ('so_number',  'nunique'),
        last_order    = ('order_date', 'max')
    )
    .reset_index()
)
df_cust_monthly['ds'] = pd.to_datetime(
    df_cust_monthly['fiscal_year'].astype(str) + '-' +
    df_cust_monthly['fiscal_month'].astype(str).str.zfill(2) + '-01'
)
print(f"Customer-monthly rows: {len(df_cust_monthly):,}")
df_cust_monthly.tail(3)

# Data Validation

In [ ]:
# ─── Kiểm tra null values ───
print("=== Null check — fact_sales ===")
print(df_fact.isnull().sum()[df_fact.isnull().sum() > 0])

In [ ]:
# ─── Visualize xu hướng tổng quan ───
df_trend = df_fact.groupby('order_date')['line_total'].sum().reset_index()
df_trend_m = df_fact.groupby(df_fact['order_date'].dt.to_period('M'))['line_total'].sum().reset_index()
df_trend_m['order_date'] = df_trend_m['order_date'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(df_trend_m['order_date'], df_trend_m['line_total'] / 1e9, color='steelblue', alpha=0.8)
ax.set_title('Tổng doanh số theo tháng — 2025 đến T3/2026', fontsize=13, fontweight='bold')
ax.set_ylabel('Doanh số (tỷ VND)')
ax.set_xlabel('Tháng')
plt.tight_layout()
plt.show()

# Save Datasets

In [ ]:
# ─── Lưu các dataset ra CSV để dùng cho C1, C2, C3 ───
df_fact.to_csv('data/fact_sales_full.csv', index=False)
df_monthly.to_csv('data/monthly_by_group.csv', index=False)
df_sku_monthly.to_csv('data/sku_monthly.csv', index=False)
df_cust_monthly.to_csv('data/customer_monthly.csv', index=False)
df_customers.to_csv('data/customers.csv', index=False)

print("✅ Saved:")
print("   data/fact_sales_full.csv       → C1, C2, C3")
print("   data/monthly_by_group.csv      → C1 (Dự báo doanh số)")
print("   data/sku_monthly.csv           → C2 (Dự báo màu sắc)")
print("   data/customer_monthly.csv      → C3 (Dự báo đại lý)")
print("   data/customers.csv             → C3 (thông tin đại lý)")